# <span style="color:red; font-size: 30px"> Assignment 9 - Due Thursday, April 16th at 8am ET</span>

<font size = "3">

This is a graded homework assignment - no part may be used within a prompt to an LLM

Complete the code cells in this Jupyter notebook, and submit the final .ipynb notebook to Gradescope.

**Please do the following in VS code before you submit**:

- Click "Clear All Outputs"

- Click "Restart"

- Run all code cells

- Save the file

**Note:** Output should match the code that appears in its corresponding cell.

<font size = "3">

Import the familiar libraries ``pandas`` and ``numpy``.

In addition, we'll import a sublibrary from the ``statsmodels`` library. We'll use this in problem (a)

In [1]:
import pandas as pd 
import numpy as np 

# for problem (a)
import statsmodels.formula.api as smf

<font size = "4">

**(a) Linear Regression**

<font size = "3">

Read in the F1 data from "results.csv" in the data folder, assigning it to a DataFrame. For F1 races, we might guess that there is a simple linear relationship between the number of points scored by a driver ("points"), the number of laps completed by the driver ("laps"), and the driver's starting position ("grid"). In particular, a better starting position and a larger number of completed laps should lead to a higher number of points.

In particular, we expect that these three variables are related by the approximate equality:

$$ p_i \approx a\cdot \ell_i + b\cdot g_i + c$$

where:

- $i$ is the index of the result (row $i$ of the DataFrame)
- $p_i$ is the number of points scored in result $i$
- $\ell_i$ is the number of laps completed in result $i$
- $g_i$ is the starting "grid" position of result $i$.
- $a$, $b$, and $c$ are the the coefficients of the linear model we need to determine.

In this model, we say that "laps" and "grid" are both **independent variables** and "points" is the **dependent variable**.

- Use ``smf.ols`` (Ordinary Least Squares) to compute the 3 coefficients of the model by running the following line of code (the names "dependent_var/independent_var1/independent_var2" are placeholder names - substitute the appropriate column names from ``df_results``):
```python
model = smf.ols(formula = "dependent_var ~ independent_var1  + independent_var2", data = df_results)
```

- Compute a Pandas ``Series`` containing the computed coefficients $a$, $b$, and $c$ from the linear model. You can do this as follows:
```python
        coeffs = model.fit().params
```
- Assign the entries of ``coeffs`` to 3 floating-point variables ``a``, ``b``, ``c`` corresponding to the coefficients of the model.

- Define two variables ``grid_val = 3`` and ``laps_val = 55``, which represent the position (grid) and completed laps, respectively, for a given driver. Using these values and the model coefficients, calculate how many points the linear model predicts for this driver, assigning it to the variable ``predicted_points``

In [2]:
# your answer here
results = pd.read_csv("data/results.csv")
model = smf.ols(formula = "points ~ laps + grid", data = results)
coeffs = model.fit().params

a = coeffs.iloc[0]
b = coeffs.iloc[1]
c = coeffs.iloc[2]

grid_val = 3
laps_val = 55

predicted_points = a + (laps_val * b) + (grid_val * c)

<font size = "4">

**(b) Recoding variables**

<font size = "3">

- Read in the file "players_15.csv" from the "data" folder and assign it to a DataFrame. (You might get a warning message, which is *not* the same things as an error message)

- The DataFrame has a column called "wage_eur" which records player wages given in Euros.

- Use the function `pandas.cut` to add a new column to the DataFrame called "wage_level" that categorizes wages as follows:

    - 'Low' if $\textrm{wage} \leq 50,000$

    - 'Medium' if $50,000 < \textrm{wage} \leq 100,000$

    - 'High' if $100,000 < \textrm{wage} \leq 200,000$

    - 'Elite' if $200,000 < \textrm{wage}$

In [3]:
# your code here
players = pd.read_csv("data/players_15.csv")
bins = [0, 50000, 100000, 200000, np.inf]
values = ["Low", "Medium", "High", "Elite"]

players["wage_level"] = pd.cut(x = players["wage_eur"], bins = bins, right = True, labels = values)

C:\Users\tucke\AppData\Local\Temp\ipykernel_28980\3070220905.py:2: DtypeWarning: Columns (104) have mixed types. Specify dtype option on import or set low_memory=False.
  players = pd.read_csv("data/players_15.csv")


<font size = "4">

**(c) Compute Cost per Rating by Wage Group**

<font size = "3">

- Continue to use the DataFrame you loaded in question (b)

- Create a new column called "cost_per_rating" which is defined as the ratio between the player's wage and the rating captured in the "overall" column:

$$\textrm{cost per rating} = \frac{\textrm{wage in euros}}{\textrm{overall rating}} $$

- Use `.groupby` to create a DataFrameGroupBy object, grouping by the "wage_level" column. (This might also give you a warning message)

- Compute the average "cost_per_rating" for each wage group, and sort the resulting Pandas Series in descending order.

In [4]:
# your code here
players["cost_per_rating"] = players["wage_eur"] / players["overall"]

new_df = players.groupby(by = "wage_level")

result = new_df["cost_per_rating"].mean()
result.sort_values(ascending = False)

C:\Users\tucke\AppData\Local\Temp\ipykernel_28980\4136538416.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  new_df = players.groupby(by = "wage_level")


wage_level
Elite     2972.550004
High      1676.011798
Medium     963.498497
Low        127.210624
Name: cost_per_rating, dtype: float64

<font size = "4">

**(d) Aggregated Statistics**

<font size = "3">

- Read in the file "monthlysalesdata.csv" in the "data" folder, assigning it to a DataFrame.

- Use `.groupby` to group the dataset by the "Category" variable

- Use the `.agg` method to create a DataFrame with the following aggregated statistics:

    1. The total number of units sold in each category (name this column "total_units")

    2. The average revenue for sales in each category (name this column "mean_revenue")

- Print the resulting DataFrame to the screen. Since the number of units is a whole number, and the mean revenue is in terms of money, ensure the printed DataFrame is rounded to two digits.

In [5]:
# your code here
sales = pd.read_csv("data/monthlysalesdata.csv")

grouped = sales.groupby(by = "Category")
final = grouped.agg(total_units = ('Units Sold', 'sum'), mean_revenue = ('Revenue ($)', 'mean'))
final["mean_revenue"] = final["mean_revenue"].round(2)
print(final)

           total_units  mean_revenue
Category                            
Audio              280       5316.67
Computing          155      41333.33
Wearables          360      18000.00


<font size = "4">

**(e) Aggregated Statistics with a user-defined function**

<font size = "3">

- Continue to the use the DataFrame created using "monthlysalesdata.csv".

- Use `.groupby` to group the dataset by the "Branch" variable

- Use the `.agg` method to create a DataFrame with the following aggregated statistics:

    1. The total revenue for each branch (name this column "branch_revenue")

    2. The average customer feedback score for each branch (name this column "mean_rating")

    3. The total number of "high" customer ratings, where a high rating is any rating $\geq 4.5$ (name this column "num_hi_ratings"). You will need to use your own custom function for this statistic.

In [6]:
# your code here
def function(data):
    output = 0
    for p in data:
        if p >= 4.5:
            output += 1
    return output

grouped2 = sales.groupby(by = "Branch")

final2 = grouped2.agg(branch_revenue = ('Revenue ($)', 'sum'), mean_rating = ('Customer Feedback (1-5)', 'mean'), num_hi_ratings = ('Customer Feedback (1-5)', function))
print(final2)

        branch_revenue  mean_rating  num_hi_ratings
Branch                                             
East             54500     4.433333               1
North            71700     4.533333               2
South            67750     4.100000               1
